# 2a. Provider catalog, policies and connectors

The Provider publishes what it offers as a DCAT catalog: datasets, their distributions (how the
data is delivered) and ODRL policies (the terms under which it may be used). A connector instance
bound to each distribution tells the Provider how to reach the real backend when a transfer
starts; in this notebook the connectors use no authentication (`NO_AUTH`).

Steps:
1. Onboarding (`setup_auth.ipynb`), guarded.
2. Resolve the Provider's main catalog and data service.
3. Create a dataset with a PULL distribution (`http+pull`) and a PUSH distribution (`http+push`).
4. Attach an ODRL offer instantiated from a policy template.
5. Attach two ODRL offers written by hand.
6. Register a PULL connector template and instantiate it on the PULL distribution.
7. Register a PUSH connector template and instantiate it on the PUSH distribution.
8. The Consumer discovers the catalog over DSP and picks the dataset and offer to negotiate.

**Runs on its own.** It leaves `target_id`, `policy_id` and `data_space_provider_dsp_endpoint`
ready for the negotiation notebooks, and sets `catalog_setup = "no_auth"` so that the transfer
notebooks know which variant is loaded. Each run creates a new dataset. `setup_catalog_policies_oauth.ipynb` is the
same flow with connectors that authenticate with a bearer token.

## 1. Prerequisites

The guard runs the onboarding notebook only when this kernel has not run it yet. `%run` on a
notebook executes its cells in this same namespace, so the URLs, DIDs and `show` it defines are
available below.

In [ ]:
if not globals().get("onboarded"):
    %run ./setup_auth.ipynb

## 2. Main catalog and data service

Every Provider starts with a `main` catalog and a `main` data service. The data service is the
Provider's DSP endpoint; distributions point to it through `dcatAccessService`.

In [ ]:
catalog = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/catalogs/main", headers=json_header))
catalog_id = catalog["id"]

In [ ]:
data_service = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/data-services/main", headers=json_header))
data_service_id = data_service["id"]

## 3. Dataset and distributions

A dataset is the unit of negotiation: a contract agreement always targets a dataset. Its
distributions are the ways to obtain it. The format of a distribution is the `format` a transfer
request asks for later:
- `http+pull`: the Consumer pulls the data from an endpoint the Provider hands over.
- `http+push`: the Provider pushes the data to a sink the Consumer names.

In [ ]:
payload = {
    "dctTitle": "My dataspace dataset",
    "catalogId": catalog_id,
}
dataset = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/datasets", headers=json_header, json=payload))
dataset_id = dataset["id"]

In [ ]:
payload = {
    "dctTitle": "Distribution of my dataset",
    "dctDescription": "Pull access to my dataset",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "http+pull",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_pull_id = distribution["id"]

In [ ]:
payload = {
    "dctTitle": "Distribution of a streaming dataset",
    "dctDescription": "Push delivery of my streaming dataset",
    "dcatAccessService": data_service_id,
    "datasetId": dataset_id,
    "dctFormats": "http+push",
}
distribution = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/distributions", headers=json_header, json=payload))
distribution_push_id = distribution["id"]

## 4. ODRL offer from a policy template

Policy templates are parameterised ODRL policies. The agent seeds `policy-1`, whose single
permission constrains `bla` to the value of the `$bla` parameter. Instantiating it attaches the
resulting offer to the dataset.

In [ ]:
templates = show(requests.get(f"{data_space_provider}/api/v1/catalog-agent/policy-templates", headers=json_header))

In [ ]:
payload = {
    "id": "policy-1",
    "version": "1.0",
    "parameters": {
        "$bla": "did:web:university.upm.es"
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research access for UPM: grants use to did:web:university.upm.es.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/policy-templates/instantiate-odrl-offer", headers=json_header, json=payload))

## 5. ODRL offers written by hand

An offer can also be posted as plain ODRL: permissions, obligations (duties) and prohibitions,
each with optional constraints. The Consumer will see all three offers on the dataset.

In [ ]:
payload = {
    "odrlOffer": {
        "permission": [{
            "action": "use",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-01-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-12-31T23:59:59Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "commercial"},
            ],
        }],
        "obligation": [{
            "action": "attribute",
            "constraint": [{"leftOperand": "recipient", "operator": "isA", "rightOperand": "public"}],
        }],
        "prohibition": [{"action": "derive", "constraint": []}],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Commercial licence for 2026: commercial use with mandatory attribution; derivation is prohibited.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

In [ ]:
payload = {
    "odrlOffer": {
        "permission": [{
            "action": "read",
            "constraint": [
                {"leftOperand": "dateTime", "operator": "gteq", "rightOperand": "2026-03-01T00:00:00Z"},
                {"leftOperand": "dateTime", "operator": "lteq", "rightOperand": "2026-09-01T00:00:00Z"},
                {"leftOperand": "purpose", "operator": "isA", "rightOperand": "research"},
                {"leftOperand": "count", "operator": "lteq", "rightOperand": "10000"},
            ],
        }],
        "obligation": [],
        "prohibition": [
            {"action": "distribute", "constraint": []},
            {"action": "reproduce", "constraint": []},
        ],
    },
    "entityId": dataset_id,
    "entityType": "Dataset",
    "description": "Research trial (Mar-Sep 2026): read only, up to 10 000 requests, research purposes; no redistribution or reproduction.",
}
policy = show(requests.post(f"{data_space_provider}/api/v1/catalog-agent/odrl-policies", headers=json_header, json=payload))

## 6. PULL connector

A connector template describes how to reach a backend, with `{{__NAME__}}` placeholders; an
instance fills the parameters and binds the template to a distribution. In PULL mode the template
describes the data access the Consumer gets once the transfer starts.

In [ ]:
payload = {
    "authentication": {"type": "NO_AUTH"},
    "interaction": {
        "mode": "PULL",
        "dataAccess": {
            "protocol": "HTTP",
            "urlTemplate": "{{__ACCESS_URL__}}",
            "method": "{{__ACCESS_METHODS__}}",
            "headers": "{{__ACCESS_HEADERS__}}"
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "ACCESS_URL", "title": "Url", "required": True},
        {"paramType": "VEC<STRING>", "name": "ACCESS_METHODS", "title": "Methods", "required": True},
        {"paramType": "MAP<STRING,STRING>", "name": "ACCESS_HEADERS", "title": "Headers", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

In [ ]:
payload = {
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_pull_id,
    "metadata": {
        "description": "Pull access to the demo backend",
        "ownerId": "tutorial"
    },
    "dryRun": False,
    "parameters": {
        "ACCESS_URL": "http://127.0.0.1:8081",
        "ACCESS_METHODS": ["GET", "HEAD"],
        "ACCESS_HEADERS": {"Content-type": "application/json"}
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))

## 7. PUSH connector

In PUSH mode the template says how to subscribe the Provider to the backend when a transfer
starts, and how to unsubscribe when it stops. `SUB_URL` and `UNSUB_URL` are filled by the
instance; the `SYS_*` and `RUNTIME_*` placeholders are filled by the agent at run time (its own
URL, the ingress of the transfer, and the subscription id the backend answered).

In [ ]:
payload = {
    "authentication": {"type": "NO_AUTH"},
    "interaction": {
        "mode": "PUSH",
        "subscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__SUB_URL__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({
                "url": "{{__SYS_OWN_URL_DOCKER__}}{{__RUNTIME_INGRESS__}}",
                "event_type": "hotel_waste_generated"
            })
        },
        "unsubscribe": {
            "protocol": "HTTP",
            "urlTemplate": "{{__UNSUB_URL__}}/{{__RUNTIME_JSON_{.subscription.data.ID}__}}",
            "method": "POST",
            "headers": {},
            "bodyTemplate": json.dumps({})
        }
    },
    "parameters": [
        {"paramType": "STRING", "name": "SUB_URL", "title": "Subscription URL", "required": True},
        {"paramType": "STRING", "name": "UNSUB_URL", "title": "Unsubscription URL", "required": True}
    ]
}
template = show(requests.post(f"{data_space_provider}/api/v1/connector/templates", headers=json_header, json=payload))

In [ ]:
payload = {
    "templateName": template["name"],
    "templateVersion": template["version"],
    "distributionId": distribution_push_id,
    "parameters": {
        "SUB_URL": "http://127.0.0.1:8082/subscriptions/subscribe",
        "UNSUB_URL": "http://127.0.0.1:8082/subscriptions/unsubscribe"
    }
}
instance = show(requests.post(f"{data_space_provider}/api/v1/connector/instances", headers=json_header, json=payload))

## 8. Catalog discovery by the Consumer (DSP 6.2.1)

The Consumer asks its own agent to fetch the Provider's catalog
(`/dsp/current/catalog/rpc/setup-catalog-request`); the agent sends the DSP `CatalogRequestMessage`
with its peer token. From the answer we keep:
- `target_id`: the `@id` of the dataset created above.
- `policy_id`: the `@id` of one of its offers. A negotiation must open on an offer the catalog
  publishes, on the dataset that publishes it.
- `data_space_provider_dsp_endpoint`: the `endpointURL` of the Provider's data service (`service`
  is a list).

In [ ]:
payload = {
    "associatedAgentPeer": data_space_provider_participant_id,
    "filter": [],
    "noCache": True,
}
provider_catalog = show(requests.post(f"{data_space_consumer}/dsp/current/catalog/rpc/setup-catalog-request", headers=json_header, json=payload))["response"]

In [ ]:
published = next(d for d in provider_catalog["dataset"] if d["@id"] == dataset_id)
target_id = published["@id"]
policy_id = published["hasPolicy"][0]["@id"]
data_space_provider_dsp_endpoint = provider_catalog["service"][0]["endpointURL"]

# Marker read by the guards of the negotiation and transfer notebooks
catalog_setup = "no_auth"

print(f"Dataset (target):      {target_id}")
print(f"Offer (policy):        {policy_id}")
print(f"Provider DSP address:  {data_space_provider_dsp_endpoint}")